# wasurenaide 音源分析 — Qwen2-Audio

MP3をアップロードすると、25秒ごとの聴感分析、BPM・低域などの数値測定、曲全体の統合評価を日本語で返します。

## iPhoneでの使い方
1. Colabのメニューから **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択します。
2. 上から順にセル左側の再生ボタンを押します。
3. 最後に表示されるURLを開き、MP3を選んで **音源を分析** を押します。

初回はモデルのダウンロードに時間がかかります。曲は25秒単位に分割し、Qwen2-Audioが得意とする30秒未満の入力にします。

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes librosa soundfile gradio
print('準備が完了しました')

In [ ]:
import gc
import numpy as np
import librosa
import torch
import gradio as gr
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError('GPUが有効ではありません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。')

MODEL_ID = 'Qwen/Qwen2-Audio-7B-Instruct'
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=quantization,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()
DEVICE = next(model.parameters()).device
SR = processor.feature_extractor.sampling_rate
print(f'Qwen2-Audioを読み込みました / GPU: {torch.cuda.get_device_name(0)} / sample rate: {SR}')

In [ ]:
DEFAULT_QUESTION = '''wasurenaideの制作監督として、この区間で実際に聞こえるものだけを日本語で分析してください。推測と聞き取れた事実を分けてください。
1. 主な楽器・音色・ノイズ
2. ドラムの型、跳ね方、ベースの種類
3. 声の高さ、ロボット感、発音、歌唱速度
4. 音の歪み、空間、低音の体感
5. フックとして記憶に残る音や言葉
6. この区間内の変化
7. 指定した可能性のある要素のうち、実際には弱いもの
聞こえない楽器を想像で追加しないでください。'''

def _processor_with_audio(text, audio):
    try:
        return processor(text=text, audio=[audio], return_tensors='pt', padding=True).to(DEVICE)
    except TypeError:
        return processor(text=text, audios=[audio], return_tensors='pt', padding=True).to(DEVICE)

def analyze_audio_chunk(audio, question, start_sec, end_sec):
    time_label = f'{start_sec:.1f}〜{end_sec:.1f}秒'
    conversation = [
        {'role': 'system', 'content': 'あなたは音楽制作と音響の批評家です。音源から確認できないことを断定しません。日本語で具体的に答えます。'},
        {'role': 'user', 'content': [
            {'type': 'audio', 'audio_url': f'local://{time_label}.wav'},
            {'type': 'text', 'text': f'対象区間: {time_label}\n{question}'},
        ]},
    ]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = _processor_with_audio(text, audio)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=380, do_sample=False)
    generated = generated[:, inputs.input_ids.shape[1]:]
    result = processor.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    del inputs, generated
    gc.collect()
    torch.cuda.empty_cache()
    return result.strip()

def text_only_summary(prompt):
    conversation = [
        {'role': 'system', 'content': 'あなたは音楽制作の統括ディレクターです。与えられた区間分析を統合し、日本語で具体的かつ率直に答えます。'},
        {'role': 'user', 'content': [{'type': 'text', 'text': prompt}]},
    ]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, return_tensors='pt', padding=True).to(DEVICE)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=700, do_sample=False)
    generated = generated[:, inputs.input_ids.shape[1]:]
    result = processor.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return result.strip()

def numeric_audio_report(y, sr):
    duration = librosa.get_duration(y=y, sr=sr)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo = float(np.ravel(tempo)[0])
    rms = librosa.feature.rms(y=y)[0]
    power = np.abs(librosa.stft(y, n_fft=2048, hop_length=512)) ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
    total = float(power.sum()) + 1e-12
    sub = float(power[freqs < 120].sum() / total * 100)
    low_mid = float(power[(freqs >= 120) & (freqs < 500)].sum() / total * 100)
    high = float(power[freqs >= 5000].sum() / total * 100)
    return {
        'duration': duration,
        'tempo': tempo,
        'rms_mean': float(rms.mean()),
        'rms_peak': float(rms.max()),
        'sub': sub,
        'low_mid': low_mid,
        'high': high,
    }

def analyze_song(audio_path, question):
    if not audio_path:
        return 'MP3またはWAVを選んでください。'
    question = (question or DEFAULT_QUESTION).strip()
    y, sr = librosa.load(audio_path, sr=SR, mono=True)
    stats = numeric_audio_report(y, sr)
    chunk_seconds = 25
    overlap_seconds = 2
    chunk_samples = int(chunk_seconds * sr)
    step_samples = int((chunk_seconds - overlap_seconds) * sr)
    analyses = []
    starts = list(range(0, len(y), step_samples))
    for index, start in enumerate(starts, 1):
        end = min(start + chunk_samples, len(y))
        if end - start < sr * 2:
            continue
        start_sec, end_sec = start / sr, end / sr
        result = analyze_audio_chunk(y[start:end], question, start_sec, end_sec)
        analyses.append(f'### 区間 {index}（{start_sec:.1f}〜{end_sec:.1f}秒）\n{result}')

    stats_text = (
        f"長さ: {stats['duration']:.1f}秒 / 推定BPM: {stats['tempo']:.1f} / "
        f"平均RMS: {stats['rms_mean']:.4f} / 最大RMS: {stats['rms_peak']:.4f} / "
        f"120Hz未満: {stats['sub']:.1f}% / 120〜500Hz: {stats['low_mid']:.1f}% / 5kHz以上: {stats['high']:.1f}%"
    )
    joined = '\n\n'.join(analyses)
    summary_prompt = f'''以下は同じ一曲を時間順に分析した結果です。矛盾があれば断定せず、複数区間で一致する観察を優先してください。

数値測定:
{stats_text}

区間分析:
{joined}

次の形式で曲全体を統合評価してください。
1. 実際に聞こえるジャンルとリズム
2. 主役の楽器・低音・ノイズ
3. ボーカルの一貫性
4. 最も強いフック
5. ヴァース、サビ、ブレイクの変化
6. 弱い点または狙いが反映されていない点
7. Sunoで次に直すべき指示を優先順に3つ
聞こえた事実と推測を分けてください。'''
    try:
        summary = text_only_summary(summary_prompt)
    except Exception as exc:
        summary = f'統合生成のみ失敗しました: {exc}。下の区間分析は利用できます。'

    return f'''# 曲全体の統合評価

## 数値測定
{stats_text}

{summary}

---
# 区間ごとの聴感分析
{joined}
'''


In [ ]:
with gr.Blocks(title='wasurenaide Qwen2-Audio Analyzer') as demo:
    gr.Markdown('# wasurenaide 音源分析\nMP3/WAVを選び、「音源を分析」を押してください。')
    audio_file = gr.Audio(sources=['upload'], type='filepath', label='分析する曲')
    question = gr.Textbox(value=DEFAULT_QUESTION, lines=10, label='Qwen2-Audioへの分析指示')
    analyze_button = gr.Button('音源を分析', variant='primary')
    output = gr.Markdown()
    analyze_button.click(analyze_song, inputs=[audio_file, question], outputs=output)

demo.queue().launch(share=True, debug=True)